# NoiseInject Quickstart

This notebook walks through the full NoiseInject workflow on **public datasets bundled with scikit-learn**, so it runs end-to-end with no extra downloads or chemistry dependencies.

It covers both task types the framework supports:

1. **Regression** &mdash; inject calibrated continuous (Gaussian-family) noise into the targets and measure how predictive performance degrades.
2. **Classification** &mdash; inject calibrated label flips and measure accuracy retention, including per-class robustness.

The pattern in both cases is the same: **calibrate** a noise level to a target effective noise, **inject** at a sweep of multiples, retrain, and **summarise** robustness with the Noise Sensitivity Index (NSI) and retention.

> Install first (from the repo root): `pip install -e .`

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from noiseInject import (
    NoiseInjectorRegression,
    NoiseInjectorClassification,
    calibrate_sigma,
    calibrate_flip_probability,
    calculate_noise_metrics,
    calculate_classification_metrics,
)

RANDOM_STATE = 42

## 1. Regression: California Housing

We load a standard public regression benchmark and hold out a clean test set. Only the **training labels** get noise injected &mdash; the test labels stay clean so we measure the effect of *learning from* noisy targets.

In [2]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"train={len(X_train)}  test={len(X_test)}  target range=[{y.min():.2f}, {y.max():.2f}]")

train=16512  test=4128  target range=[0.15, 5.00]


### Calibrate, then sweep noise levels

`calibrate_sigma` finds the base `sigma` that produces a target *effective* noise (here 10% of the label standard deviation) for the chosen strategy. We then sweep multiples of that calibrated sigma; the `0.0` level is the clean baseline.

In [3]:
strategy = 'legacy'  # homogeneous Gaussian; try 'hetero', 'quantile', 'outlier', ...

sigma = calibrate_sigma(
    y_train, target_effective_noise=0.1, strategy=strategy, random_state=RANDOM_STATE
)
print(f"Calibrated sigma for 10% effective noise: {sigma:.4f}")

injector = NoiseInjectorRegression(strategy, random_state=RANDOM_STATE)

predictions = {}
for mult in [0.0, 1.0, 2.0, 3.0, 4.0]:
    y_noisy = y_train if mult == 0.0 else injector.inject(y_train, sigma * mult)
    model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_noisy)
    predictions[sigma * mult] = model.predict(X_test)

Calibrated sigma for 10% effective noise: 0.1562


In [4]:
per_sigma_df, summary_df = calculate_noise_metrics(y_test, predictions, metrics=['r2', 'rmse', 'mae'])

print('Per-sigma performance:')
print(per_sigma_df.to_string(index=False))

s = summary_df.iloc[0]
print(f"\nBaseline R2 : {s['baseline_r2']:.4f}")
print(f"NSI (R2)    : {s['nsi_r2']:.4f}   (slope of R2 vs noise; more negative = less robust)")
print(f"Retention   : {s['retention_pct_r2']:.1f}%")

Per-sigma performance:
  sigma       r2     rmse      mae
0.00000 0.805123 0.505340 0.327543
0.15625 0.804882 0.505653 0.330778
0.31250 0.795947 0.517101 0.343254
0.46875 0.791202 0.523078 0.353392
0.62500 0.784066 0.531941 0.366411

Baseline R2 : 0.8051
NSI (R2)    : -0.0357   (slope of R2 vs noise; more negative = less robust)
Retention   : 97.4%


## 2. Classification: Wine (3 classes)

Same workflow with label flips instead of continuous noise. We use a 3-class dataset so the per-class robustness breakdown is meaningful.

In [5]:
from sklearn.datasets import load_wine

wine = load_wine()
Xc, yc = wine.data, wine.target

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.3, random_state=RANDOM_STATE, stratify=yc
)
print(f"train={len(Xc_train)}  test={len(Xc_test)}  classes={np.unique(yc)}")

train=124  test=54  classes=[0 1 2]


In [6]:
flip_prob = calibrate_flip_probability(yc_train, target_flip_rate=0.1, random_state=RANDOM_STATE)
print(f"Calibrated flip probability for 10% flip rate: {flip_prob:.4f}")

clf_injector = NoiseInjectorClassification('uniform', random_state=RANDOM_STATE)

clf_predictions = {}
for mult in [0.0, 1.0, 2.0, 3.0]:
    y_noisy = yc_train if mult == 0.0 else clf_injector.inject(yc_train, flip_prob * mult)
    model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(Xc_train, y_noisy)
    clf_predictions[flip_prob * mult] = model.predict(Xc_test)

Calibrated flip probability for 10% flip rate: 0.0938


In [7]:
per_flip_df, clf_summary_df, per_class_df = calculate_classification_metrics(yc_test, clf_predictions)

print('Per-flip-level performance:')
print(per_flip_df.to_string(index=False))

cs = clf_summary_df.iloc[0]
print(f"\nNSI (accuracy) : {cs['nsi_accuracy']:.4f}")
print(f"Retention      : {cs['retention_pct_accuracy']:.1f}%")

print('\nPer-class robustness:')
print(per_class_df.to_string(index=False))

Per-flip-level performance:
 flip_prob  accuracy  precision_macro  recall_macro  f1_macro  precision_weighted  recall_weighted  f1_weighted
   0.00000  1.000000         1.000000      1.000000  1.000000            1.000000         1.000000     1.000000
   0.09375  0.962963         0.965608      0.965608  0.965608            0.962963         0.962963     0.962963
   0.18750  0.925926         0.927451      0.936508  0.926535            0.933987         0.925926     0.924159
   0.28125  0.944444         0.952381      0.939683  0.942419            0.952381         0.944444     0.945033

NSI (accuracy) : -0.2173
Retention      : 94.4%

Per-class robustness:
 class  flip_prob  f1_score  precision   recall  support
     0    0.00000  1.000000   1.000000 1.000000       18
     1    0.00000  1.000000   1.000000 1.000000       21
     2    0.00000  1.000000   1.000000 1.000000       15
     0    0.09375  0.944444   0.944444 0.944444       18
     1    0.09375  0.952381   0.952381 0.952381       2

## Where to go next

- Swap `strategy` to compare noise models: regression supports `legacy`, `quantile`, `threshold`, `outlier`, `hetero`, `valprop`; classification supports `uniform`, `class_imbalance`, `binary_asymmetric`, `instance_noise`, `class_dependent`, `confusion_directed`.
- Any scikit-learn-compatible estimator works in place of the random forests above.
- For chemistry-specific workflows (QM9 descriptors, MoleculeNet GNN embeddings, Tox21 toxicity), see the scripts in `examples/`.